![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)
# Google ADK Agent with RedisVL MCP and Gemini

<a href="https://colab.research.google.com/github/redis-developer/redis-ai-resources/blob/main/python-recipes/MCP/00_google_adk_redisvl_mcp_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

This notebook shows how to build a [Google ADK](https://google.github.io/adk-docs/) agent that connects to Redis through the [RedisVL MCP](https://docs.redisvl.com/en/stable/user_guide/how_to_guides/mcp.html) server. We will bootstrap a small movie catalog in Redis, generate an MCP config, wire ADK to the server, and run grounded movie-recommendation prompts.

### Key Concepts

Before diving into code, here is a quick orientation on the three technologies this recipe combines:

**[Model Context Protocol (MCP)](https://modelcontextprotocol.io/)**

MCP is an open standard that lets AI agents discover and call external tools through a uniform interface. An MCP *server* advertises a set of typed tools (search, upsert, etc.), and an MCP *client* inside the agent framework calls them. Communication happens over a *transport* -- common options include **stdio** (subprocess over stdin/stdout) and **Streamable HTTP** (an HTTP endpoint). This notebook uses Streamable HTTP for compatibility with notebook environments.

**[Google Agent Development Kit (ADK)](https://google.github.io/adk-docs/)**

ADK is Google's open-source framework for building AI agents in Python. It provides `LlmAgent` for wrapping a model with tools, `Runner` for executing multi-turn conversations, and `McpToolset` for connecting to MCP servers. ADK handles the MCP client lifecycle automatically -- it starts the subprocess, discovers the available tools, and maps them into the agent's tool list.

**[RedisVL MCP Server](https://docs.redisvl.com/en/stable/user_guide/how_to_guides/mcp.html)**

RedisVL ships an MCP server (`rvl mcp`) that exposes Redis search indexes as MCP tools. You provide a YAML config that binds an index to the server, and it advertises tools such as `search-records` and `upsert-records`. This gives any MCP-compatible agent framework (ADK, Claude Desktop, Cursor, etc.) direct access to Redis-backed retrieval without custom glue code.


### Architecture

The data flow in this recipe is:

```
User prompt
    |
    v
Google ADK Agent (Gemini API)
    |  calls MCP tool via Streamable HTTP
    v
RedisVL MCP Server (background process on localhost)
    |  executes hybrid search
    v
Redis (vector + full-text index)
    |  returns ranked results
    v
RedisVL MCP Server
    |  returns structured results
    v
Google ADK Agent
    |  grounds answer in evidence
    v
Final response to user
```

The agent never talks to Redis directly. All data access goes through the MCP tool interface, which means the same Redis index could be shared with other MCP-compatible clients without any code changes.

> **Note on transport choice:** MCP supports stdio (subprocess communication) and HTTP-based transports (Streamable HTTP, SSE). This notebook uses **Streamable HTTP** because notebook environments like Colab and Jupyter replace `sys.stderr`/`sys.stdout` with custom stream objects that break stdio subprocess communication. Streamable HTTP runs the MCP server as an independent process and connects over a normal TCP socket, which works reliably everywhere.


## What We'll Build

This tutorial is for readers who already know the basics of Redis and Python and want to see a minimal MCP-powered agent pattern end to end.

**By the end you will be able to:**
- load a small JSON movie dataset into a Redis index with RedisVL
- configure RedisVL MCP to expose that index in read-only mode
- connect a Google ADK `LlmAgent` to the MCP server while using the Gemini API for the model
- inspect the structured Redis-backed evidence the agent retrieved before it answered

## Outline

1. Install packages.
2. Configure the Gemini API key and Redis.
3. Load the movie sample data and create a Redis index.
4. Generate an MCP YAML config for RedisVL.
5. Configure the MCP toolset and build the ADK agent.
6. Validate the MCP connection and define interaction helpers.
7. Run a first grounded prompt.
8. Try more prompts.
9. Exercise.


## 1. Install Packages

We will use:
- `redisvl[mcp,sentence-transformers]` for indexing, local embeddings, and the MCP server
- `google-adk` for the agent runtime (connects to the Gemini API with an API key)


In [ ]:
%pip install -q "redisvl[mcp,sentence-transformers]>=0.17.1" "google-adk>=1.0.0" pandas nest_asyncio


### Optional: Install Redis Stack in Colab

If you already have a Redis deployment with Search enabled, you can skip this step and use your own connection details instead.


In [ ]:
# NBVAL_SKIP
!git clone https://github.com/redis-developer/redis-ai-resources.git temp_repo
!mv temp_repo/python-recipes/MCP/resources .
!rm -rf temp_repo

In [ ]:
# NBVAL_SKIP
%%sh
sudo apt-get install lsb-release curl gpg
curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
sudo chmod 644 /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
sudo apt-get update
sudo apt-get install redis

redis-server --version
redis-server --daemonize yes --loadmodule /usr/lib/redis/modules/redisearch.so

## 2. Configure the Gemini API Key and Redis

This notebook uses the **Gemini API** (via a simple API key) for the agent model. Redis indexing and RedisVL MCP query embeddings stay local with `HFTextVectorizer`, which keeps the retrieval setup simple and reproducible.

You can get a free Gemini API key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).


In [1]:
# NBVAL_SKIP
import os
import sys
import json
import uuid
import warnings
from getpass import getpass
from pathlib import Path

import nest_asyncio
import pandas as pd
import yaml
from IPython.display import display

warnings.filterwarnings("ignore")
nest_asyncio.apply()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass("GOOGLE_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

REDIS_URL = os.getenv("REDIS_URL")
if not REDIS_URL:
    REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
    REDIS_PORT = os.getenv("REDIS_PORT", "6379")
    REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")
    auth_part = f":{REDIS_PASSWORD}@" if REDIS_PASSWORD else ""
    REDIS_URL = f"redis://{auth_part}{REDIS_HOST}:{REDIS_PORT}"

print({"redis_url": REDIS_URL})


{'redis_url': 'redis://localhost:6379'}


In [2]:
from redis import Redis
from redisvl.index import SearchIndex
from redisvl.query import AggregateHybridQuery
from redisvl.utils.vectorize import HFTextVectorizer

redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()


True

## 3. Load the Sample Movie Data

For this tutorial we use a small movie catalog with titles, genres, ratings, and short plot descriptions. It is compact enough to inspect by eye, but still rich enough to show how Redis-backed retrieval changes an agent's answers.


In [3]:
movies_path = Path("resources") / "movies.json"
with open(movies_path, "r", encoding="utf-8") as f:
    movies = json.load(f)


In [4]:
movies_df = pd.DataFrame(movies)
print(f"Loaded {len(movies_df)} movie entries")
display(movies_df.head())


Loaded 20 movie entries


,id,title,genre,rating,description
0,1,Explosive Pursuit,action,7,A daring cop chases a notorious criminal acros...
1,2,Skyfall,action,8,James Bond returns to track down a dangerous n...
2,3,Fast & Furious 9,action,6,Dom and his crew face off against a high-tech ...
3,4,Black Widow,action,7,Natasha Romanoff confronts her dark past and f...
4,5,John Wick,action,8,A retired hitman seeks vengeance against those...


### Note on the `id` Field

RedisVL MCP reserves `id` for its own response envelope, so we rename the dataset's `id` field to `movie_id` before loading records into Redis. That small change keeps the schema compatible with the MCP server while preserving the original identifier.


In [5]:
INDEX_NAME = "adk-movies"
INDEX_PREFIX = "movie"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

vectorizer = HFTextVectorizer(model=EMBEDDING_MODEL)

schema = {
    "index": {
        "name": INDEX_NAME,
        "prefix": INDEX_PREFIX,
        "storage_type": "hash",
    },
    "fields": [
        {"name": "movie_id", "type": "tag"},
        {"name": "title", "type": "text"},
        {"name": "genre", "type": "tag"},
        {"name": "rating", "type": "numeric"},
        {"name": "description", "type": "text"},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "algorithm": "flat",
                "dims": vectorizer.dims,
                "distance_metric": "cosine",
                "datatype": "float32",
            },
        },
    ],
}

index = SearchIndex.from_dict(schema, redis_url=REDIS_URL)
index.create(overwrite=True, drop=True)

records = []
for movie in movies:
    record = {
        "movie_id": movie["id"],
        "title": movie["title"],
        "genre": movie["genre"],
        "rating": movie["rating"],
        "description": movie["description"],
    }
    record["embedding"] = vectorizer.embed(record["description"], as_buffer=True)
    records.append(record)

loaded_keys = index.load(records)
print(f"Loaded {len(loaded_keys)} movie records into Redis index '{INDEX_NAME}'.")
display(pd.DataFrame(records)[["movie_id", "title", "genre", "rating"]].head())


Loaded 20 movie records into Redis index 'adk-movies'.


,movie_id,title,genre,rating
0,1,Explosive Pursuit,action,7
1,2,Skyfall,action,8
2,3,Fast & Furious 9,action,6
3,4,Black Widow,action,7
4,5,John Wick,action,8


### Sanity-Check the Redis Index

Before adding MCP and ADK, it helps to confirm that hybrid search over the Redis index returns sensible matches.


In [6]:
user_query = "revenge-driven action story"
embedded_query = vectorizer.embed(user_query, as_buffer=True)

hybrid_query = AggregateHybridQuery(
    text=user_query,
    text_field_name="description",
    vector=embedded_query,
    vector_field_name="embedding",
    num_results=3,
    return_fields=["movie_id", "title", "genre", "rating", "description"],
)

sanity_results = index.query(hybrid_query)
sanity_df = pd.DataFrame(sanity_results)
display(sanity_df[["title", "genre", "rating", "hybrid_score"]])


,title,genre,rating,hybrid_score
0,John Wick,action,8,0.510036081076
1,Gladiator,action,8,0.499173808098
2,Despicable Me,comedy,7,0.485119789839


In [22]:
from redisvl.mcp import load_mcp_config

Path("tmp").mkdir(exist_ok=True)
mcp_config_path = Path("tmp/google_adk_movies_mcp.yaml").resolve()

mcp_config = {
    "server": {
        "redis_url": REDIS_URL,
    },
    "indexes": {
        "movies": {
            "redis_name": INDEX_NAME,
            "vectorizer": {
                "class": "HFTextVectorizer",
                "model": EMBEDDING_MODEL,
            },
            "search": {
                "type": "vector",
            },
            "runtime": {
				"text_field_name": "description",
                "vector_field_name": "embedding",
                "default_embed_text_field": "description",
                "default_limit": 5,
                "max_limit": 10,
                "max_result_window": 100,
                # The first search-records call loads the embedding model into
                # memory, which can take 30+ seconds. Give it plenty of room.
                "request_timeout_seconds": 120,
                "startup_timeout_seconds": 120,
            },
        }
    },
}

mcp_config_path.write_text(yaml.safe_dump(mcp_config, sort_keys=False), encoding="utf-8")
validated_config = load_mcp_config(str(mcp_config_path))

print(f"Wrote MCP config to {mcp_config_path.resolve()}")
print({
    "binding_id": validated_config.binding_id,
    "redis_name": validated_config.redis_name,
    "search_type": validated_config.binding.search.type,
})
print(mcp_config_path.read_text(encoding="utf-8"))


Wrote MCP config to /Users/vishal.bala/PycharmProjects/redis-ai-resources/python-recipes/MCP/tmp/google_adk_movies_mcp.yaml
{'binding_id': 'movies', 'redis_name': 'adk-movies', 'search_type': 'vector'}
server:
  redis_url: redis://localhost:6379
indexes:
  movies:
    redis_name: adk-movies
    vectorizer:
      class: HFTextVectorizer
      model: sentence-transformers/all-MiniLM-L6-v2
    search:
      type: vector
    runtime:
      text_field_name: description
      vector_field_name: embedding
      default_embed_text_field: description
      default_limit: 5
      max_limit: 10
      max_result_window: 100
      request_timeout_seconds: 120
      startup_timeout_seconds: 120



## 5. Configure the MCP Toolset and Build the ADK Agent

This section sets up the Google ADK side. We will:
1. Start the RedisVL MCP server as a background HTTP process.
2. Create a `McpToolset` that connects to the server over Streamable HTTP.
3. Define an `LlmAgent` that uses the toolset for retrieval and the Gemini API for generation.
4. Set up a `Runner` to manage multi-turn conversation state.

> **Why Streamable HTTP instead of stdio?** MCP supports two transport modes: stdio (subprocess communication over stdin/stdout) and Streamable HTTP. Notebook environments like Colab and Jupyter replace `sys.stderr` and `sys.stdout` with custom stream objects that lack `fileno()` support, which breaks stdio transport at the subprocess level. Streamable HTTP avoids this entirely by running the MCP server as an independent HTTP process and connecting over a normal TCP socket.

> **Troubleshooting:** The `uvx` command comes from the [uv](https://docs.astral.sh/uv/) package manager. If it is not on your PATH, install it with `pip install uv` or see the [uv installation docs](https://docs.astral.sh/uv/getting-started/installation/). If Gemini API calls fail, verify your `GOOGLE_API_KEY` is valid at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).

### 5a. Override the Search Tool Description

By default the `search-records` tool has a generic description. The model may hallucinate field names (e.g., `name`, `year`) that don't exist in the index. We override the tool description via the `REDISVL_MCP_TOOL_SEARCH_DESCRIPTION` environment variable so the model knows exactly which fields and filter operators are available.


In [29]:
SEARCH_TOOL_DESCRIPTION = "Search the movie catalog in Redis. Available fields: movie_id (tag), title (text), genre (tag), rating (numeric), description (text). Only use these field names in return_fields and filters."
os.environ["REDISVL_MCP_TOOL_SEARCH_DESCRIPTION"] = SEARCH_TOOL_DESCRIPTION


### 5b. Start the RedisVL MCP Server Over Streamable HTTP

The MCP server needs to be running before the agent can connect. It listens on `http://127.0.0.1:8000/mcp` and serves the RedisVL tools over Streamable HTTP.

**Option A -- Start from a terminal (recommended for local development):**

Open a separate terminal, `cd` to this notebook's directory, and run:

```bash
export REDISVL_MCP_TOOL_SEARCH_DESCRIPTION="Search the movie catalog in Redis. Available fields: movie_id (tag), title (text), genre (tag), rating (numeric), description (text). Only use these field names in return_fields and filters."

uvx --from "redisvl[mcp,sentence-transformers]" rvl mcp \
    --config tmp/google_adk_movies_mcp.yaml \
    --read-only \
    --transport streamable-http \
    --host 127.0.0.1 \
    --port 8000
```

You should see output indicating the server is running (e.g., `Uvicorn running on http://127.0.0.1:8000`). Leave the terminal open, then **skip the next code cell** and continue from step 5c.

**Option B -- Start from the notebook (required for Colab):**

The next cell launches the server as a background subprocess. This is the only option in Colab where there is no separate terminal.


In [ ]:
# NBVAL_SKIP
# Skip this cell if you started the server from a terminal (Option A).
import subprocess
import time
import requests

MCP_HOST = "127.0.0.1"
MCP_PORT = 8000
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"

mcp_server_command = [
    "uvx",
    "--from",
    "redisvl[mcp,sentence-transformers,nltk]",
    "rvl",
    "mcp",
    "--config",
    str(mcp_config_path),
    "--read-only",
    "--transport",
    "streamable-http",
    "--host",
    MCP_HOST,
    "--port",
    str(MCP_PORT),
]

# Pass the custom tool description to the subprocess environment.
mcp_env = os.environ.copy()
mcp_env["REDISVL_MCP_TOOL_SEARCH_DESCRIPTION"] = SEARCH_TOOL_DESCRIPTION

# Start the MCP server as a background process.
mcp_process = subprocess.Popen(
    mcp_server_command,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=mcp_env,
)

# Wait for the server to become ready (may take a while on first run
# while uvx downloads and installs redisvl[mcp]).
for attempt in range(90):
    try:
        requests.post(MCP_URL, json={"jsonrpc": "2.0", "method": "initialize", "id": 0}, timeout=2)
        print(f"MCP server is ready (PID {mcp_process.pid}).")
        break
    except requests.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError(
        "MCP server did not start within 90 s. "
        "Check that `uvx` is installed and `redisvl[mcp]` is available."
    )


### 5c. Connect ADK to the MCP Server

Now that the server is running, we point `McpToolset` at it using `StreamableHTTPConnectionParams`. The `tool_filter` restricts which MCP tools the agent can call -- here we only expose `search-records` since this is a read-only recipe.

If you started the server from a terminal (Option A), set `MCP_URL` here before proceeding:


In [18]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from google.genai import types

# If you started the server from a terminal (Option A) and skipped the cell
# above, make sure MCP_URL is defined:
if "MCP_URL" not in dir():
    MCP_URL = "http://127.0.0.1:8000/mcp"

APP_NAME = "redisvl_mcp_movies_app"
USER_ID = "notebook_user"

# Using Gemini 2.5 Flash -- a fast, cost-effective model that supports tool calling.
# See https://ai.google.dev/gemini-api/docs/models for available model names.
MODEL_NAME = "gemini-2.5-flash"

toolset = McpToolset(
    connection_params=StreamableHTTPConnectionParams(url=MCP_URL),
    # Only expose the search tool. Remove this filter to also enable upsert-records, etc.
    tool_filter=["search-records"],
)


### 5d. Define the ADK Agent

The `LlmAgent` wraps the model with the MCP toolset. The `instruction` string is the system prompt -- it tells the model to always call the search tool before answering and to ground responses in the retrieved evidence.


In [19]:
root_agent = LlmAgent(
    model=MODEL_NAME,
    name="movie_mcp_agent",
    instruction=(
        "You are a movie recommendation assistant with access to a movie catalog."
    ),
    tools=[toolset],
)


### 5e. Create the Session Service and Runner

ADK uses a `Runner` to execute agent turns and a `SessionService` to manage conversation state. `InMemorySessionService` is sufficient for a notebook -- in production you would use a persistent store.


In [20]:
session_service = InMemorySessionService()
runner = Runner(
    app_name=APP_NAME,
    agent=root_agent,
    session_service=session_service,
)

print(f"Agent ready: model={MODEL_NAME}, MCP server={MCP_URL}")


Agent ready: model=gemini-2.5-flash, MCP server=http://127.0.0.1:8000/mcp


## 6. Validate the MCP Connection and Define Helpers

Before running full prompts, let's verify that the MCP server starts correctly and exposes the expected tools. Then we define helper functions for interacting with the agent and extracting structured evidence from MCP tool responses.

### 6a. List Available MCP Tools

This cell queries the running MCP server and prints the tools it advertises. If this fails, check that the server started correctly in step 5a.


In [23]:
# NBVAL_SKIP
tools = await toolset.get_tools()
print(f"MCP server connected -- {len(tools)} tool(s) available:")
for tool in tools:
    print(f"  - {tool.name}: {getattr(tool, 'description', 'no description')[:80]}")


MCP server connected -- 1 tool(s) available:
  - search-records: Search the movie catalog in Redis. Available fields: movie_id (tag), title (text


### 6b. Evidence Extraction Helpers

The MCP server returns structured JSON payloads inside the ADK event stream. These helpers extract and flatten the search results into a pandas DataFrame so we can inspect exactly which records the agent used.

- **`_model_dump`** -- converts Pydantic models to plain dicts for easier inspection.
- **`_find_search_payloads`** -- recursively walks the tool response tree looking for objects that contain `search_type` and `results` keys (the RedisVL MCP search response shape).
- **`search_evidence_dataframe`** -- combines the above to produce a flat DataFrame of matched records with their scores.


In [24]:
def _model_dump(value):
    """Convert a Pydantic model to a JSON-serializable dict, or return the value as-is."""
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json", exclude_none=True)
    return value


def _find_search_payloads(value):
    """Recursively find RedisVL MCP search response payloads in a nested structure."""
    payloads = []
    if isinstance(value, dict):
        if "search_type" in value and "results" in value:
            payloads.append(value)
        for nested_value in value.values():
            payloads.extend(_find_search_payloads(nested_value))
    elif isinstance(value, list):
        for item in value:
            payloads.extend(_find_search_payloads(item))
    return payloads


def search_evidence_dataframe(tool_responses):
    """Extract the first search payload from tool responses and return it as a DataFrame."""
    payloads = _find_search_payloads(tool_responses)
    if not payloads:
        return None

    rows = []
    for item in payloads[0]["results"]:
        rows.append(
            {
                **item.get("record", {}),
                "score": item.get("score"),
                "score_type": item.get("score_type"),
            }
        )
    return pd.DataFrame(rows)


### 6c. The `ask_agent` Interaction Loop

This async function sends a single prompt to the agent, collects the final text response plus any MCP tool calls, and returns them in a dict. Each call creates a fresh session so prompts are independent.


In [25]:
async def ask_agent(query: str, session_id: str | None = None):
    """Send a query to the agent and return the answer plus raw tool interactions."""
    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id or f"session-{uuid.uuid4().hex[:8]}",
        state={},
    )

    user_message = types.Content(role="user", parts=[types.Part(text=query)])
    final_response = "No final response captured."
    tool_calls = []      # outgoing calls (arguments the model sent)
    tool_responses = []  # incoming responses (results from the MCP server)

    async for event in runner.run_async(
        session_id=session.id,
        user_id=session.user_id,
        new_message=user_message,
    ):
        # Capture outgoing tool calls (what the model asked for).
        if event.content and event.content.parts:
            for part in event.content.parts:
                fc = getattr(part, "function_call", None)
                if fc:
                    tool_calls.append(_model_dump(fc))

        # Capture incoming tool responses (what the MCP server returned).
        for response in event.get_function_responses():
            tool_responses.append(_model_dump(response))

        if event.is_final_response() and event.content and event.content.parts:
            text_parts = [
                part.text for part in event.content.parts if getattr(part, "text", None)
            ]
            if text_parts:
                final_response = "\n".join(text_parts)

    return {
        "query": query,
        "answer": final_response,
        "tool_calls": tool_calls,
        "tool_responses": tool_responses,
    }


## 7. Run a First Grounded Prompt

The next cell asks the ADK agent for a recommendation, then prints both the natural-language answer and the structured Redis evidence returned through MCP.


In [26]:
# NBVAL_SKIP
result = await ask_agent("Recommend an action movie with gadgets and advanced technology.")
print("=== Agent Answer ===")
print(result["answer"])

# Show what the model sent to the tool (useful for debugging bad field names).
if result["tool_calls"]:
    print("\n=== Tool Calls (outgoing) ===")
    print(json.dumps(result["tool_calls"], indent=2))

# Show the structured evidence the agent grounded its answer in.
evidence_df = search_evidence_dataframe(result["tool_responses"])
if evidence_df is not None:
    print("\n=== Evidence from Redis ===")
    display(evidence_df[["title", "genre", "rating", "score", "score_type", "description"]])
else:
    # If no evidence was extracted, show the raw tool responses for debugging.
    print("\n=== Raw Tool Responses ===")
    print(json.dumps(result["tool_responses"], indent=2))


=== Agent Answer ===
I recommend Fast & Furious 9. It's an action movie where "Dom and his crew face off against a high-tech enemy with advanced weapons and technology."

=== Tool Calls (outgoing) ===
[
  {
    "id": "adk-aa16375c-c682-447e-8f87-e8f9ce5afe7d",
    "args": {
      "filter": "@genre:{action}",
      "return_fields": [
        "title",
        "genre",
        "rating",
        "description"
      ],
      "query": "movie gadgets \"advanced technology\"",
      "limit": 1
    },
    "name": "search-records"
  }
]

=== Evidence from Redis ===


,title,genre,rating,score,score_type,description
0,Fast & Furious 9,action,6,0.629092,vector_distance_normalized,Dom and his crew face off against a high-tech ...


## 8. Try More Prompts

These examples stay close to the movie descriptions in the sample dataset, which makes it easy to see whether the agent is grounding correctly.


In [27]:
# NBVAL_SKIP
demo_queries = [
    "Find a funny family movie with superhero elements.",
    "Which movie best matches a revenge-driven action story?",
    "Suggest a movie about spies, technology, and a dangerous mission.",
]

for prompt in demo_queries:
    demo_result = await ask_agent(prompt)
    print(f"\nPrompt: {prompt}")
    print(demo_result["answer"])



Prompt: Find a funny family movie with superhero elements.
The Incredibles sounds like a great fit! It's a comedy with a family of superheroes. Its description says: "A family of undercover superheroes, while trying to live the quiet suburban life, are forced into action to save the world. Bob Parr (Mr. Incredible) and his wife Helen (Elastigirl) were among the world's greatest crime fighters, but now they must assume civilian identities and retreat to the suburbs to live a 'normal' life with their three children. However, the family's desire to help the world pulls them back into action when they face a new and dangerous enemy." It has a rating of 8.

Prompt: Which movie best matches a revenge-driven action story?
The movie "John Wick" matches your request for a revenge-driven action story. The description states: "A retired hitman seeks vengeance against those who wronged him, leaving a trail of destruction in his wake."

Prompt: Suggest a movie about spies, technology, and a danger

## 9. Exercise

Try changing the agent instruction so it always returns exactly **two** movie suggestions with one sentence of justification each. Then compare how that changes the answers for a comedy-focused prompt.


In [ ]:
# Your turn:
# 1. Update `root_agent` so the answer format is stricter.
# 2. Re-run a prompt such as:
#    "Recommend a lighthearted animated movie for a family movie night."
# 3. Inspect whether the MCP evidence still matches the final answer.


## Cleanup

Close the MCP toolset, terminate the background MCP server process (if started from the notebook), and drop the Redis index. If you started the server from a terminal (Option A), stop it there with Ctrl-C.


In [28]:
# NBVAL_SKIP
await toolset.close()

# Terminate the background server if it was started from the notebook (Option B).
if "mcp_process" in dir():
    mcp_process.terminate()
    mcp_process.wait(timeout=5)
    print(f"MCP server process (PID {mcp_process.pid}) terminated.")

index.delete(drop=True)


---

# Recap

## What We Built

1. **Redis-backed movie index** -- loaded a JSON dataset into a RedisVL search index with vector, text, tag, and numeric fields.
2. **MCP config** -- wrote a YAML file that binds the index to the RedisVL MCP server with hybrid search settings.
3. **Google ADK agent** -- connected an `LlmAgent` to the MCP server over Streamable HTTP using `McpToolset`, with the Gemini API as the model backend.
4. **Evidence inspection** -- extracted structured search results from MCP tool responses so we could verify grounding.

## Why Redis for MCP?

- **Sub-millisecond retrieval** -- Redis keeps the index in memory, so even under agent-loop latency budgets, search stays fast.
- **Hybrid search** -- combining BM25 full-text scoring with vector similarity in a single query gives better relevance than either alone.
- **Unified data layer** -- the same Redis instance can serve caching, session state, and retrieval, reducing infrastructure complexity.
- **MCP compatibility** -- RedisVL's built-in MCP server means any MCP-compatible client (ADK, Claude Desktop, Cursor, etc.) can access the same index with zero custom code.

## Common Pitfalls

- **Reserved `id` field** -- RedisVL MCP uses `id` in its response envelope. If your source data has an `id` column, rename it (e.g., to `movie_id`) before loading.
- **Missing `uvx`** -- the `uvx` command comes from the [uv](https://docs.astral.sh/uv/) package manager. If it is not on your PATH, install it with `pip install uv` or see the [uv installation docs](https://docs.astral.sh/uv/getting-started/installation/).
- **Gemini API key** -- if API calls fail, verify your key is valid at [aistudio.google.com/apikey](https://aistudio.google.com/apikey). Free-tier keys have rate limits.

## Next Steps

- Add a second notebook that enables `upsert-records` for ingestion workflows.
- Switch from the movie dataset to a domain-specific JSON corpus.
- Compare direct RedisVL querying with MCP-mediated retrieval side by side.

**Want to learn more?**
1. [RedisVL MCP documentation](https://docs.redisvl.com/en/stable/user_guide/how_to_guides/mcp.html)
2. [Google ADK MCP tools guide](https://google.github.io/adk-docs/tools/mcp-tools/)
3. [MCP protocol specification](https://modelcontextprotocol.io/)
4. [Redis AI resources repository](https://github.com/redis-developer/redis-ai-resources)
